In [16]:
import os
import pandas as pd
import numpy as np
os.chdir("../website")
from models import Database
from dotenv import load_dotenv
load_dotenv()

db = Database(
    host=os.environ["HOST"],
    port=os.environ["PORT"],
    database=os.environ["DATABASE"],
    user=os.environ["USER"],
    password=os.environ["PASSWORD"]
    )

In [17]:
input_file = r'C:\Users\keong\Downloads\STOCK 26SEP2025.xlsx'
output_file = r'C:\Users\keong\Downloads\STOCK 26SEP2025_no_returned.xlsx'

In [18]:
df = pd.read_excel(input_file)
df = df.loc[df['stk_returned']!=1]
distinct_pur = db.select(table='purchase',columns=['pur_id','pur_code','pur_gold_cost','pur_gold_cost_999','pur_date'])
merged = pd.merge(df,distinct_pur,on='pur_code',how='left')
## stk_gold_cost conditions
conditions = [
    merged['stk_gold_type'] == 916,
    merged['stk_gold_type'] == 999
]
choices = [
    merged['pur_gold_cost'],
    merged['pur_gold_cost_999']
]

merged['stk_pur_id'] = merged['pur_id']
merged['stk_gold_cost'] = np.select(conditions, choices, default=None)
merged['stk_pur_date'] = merged['pur_date']
merged['stk_status'] = 'IN STOCK'
merged.drop(columns=['pur_id','pur_gold_cost','pur_code','pur_gold_cost_999','pur_date'],inplace=True)
# merged.drop(columns=['pur_id','pur_gold_cost','pur_gold_cost_999','pur_date'],inplace=True)
merged.to_excel(output_file,index=False)